# Task 1: News Topic Classifier Using BERT

**Objective:** Fine-tune BERT (`bert-base-uncased`) to classify news headlines into 4 topic categories.

**Dataset:** AG News Dataset via Hugging Face Datasets

**Categories:** World (0) | Sports (1) | Business (2) | Sci/Tech (3)

**Skills:** NLP · Transfer Learning · Fine-tuning · Gradio Deployment

## Step 1: Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'transformers', 'datasets', 'torch', 'scikit-learn',
    'gradio', 'accelerate', 'evaluate'
]
for pkg in packages:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All dependencies ready!')

## Step 2: Import Libraries

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## Step 3: Load the AG News Dataset

In [ ]:
print('Loading AG News dataset...')
raw = load_dataset('ag_news')

# AG News has 4 classes (1-indexed in the raw data, 0-indexed after load)
LABEL_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']
NUM_LABELS  = 4

print(f'Train size : {len(raw["train"]):,}')
print(f'Test size  : {len(raw["test"]):,}')
print(f'Features   : {raw["train"].features}')

# Preview a few examples
for i in range(3):
    ex = raw['train'][i]
    print(f'  [{LABEL_NAMES[ex["label"]]}] {ex["text"][:90]}...')

In [ ]:
# Use a subset for faster fine-tuning (increase for better accuracy)
# 4,000 train + 800 validation + 800 test
TRAIN_SIZE = 4000
EVAL_SIZE  = 800

train_small = raw['train'].shuffle(seed=42).select(range(TRAIN_SIZE))
test_small  = raw['test'].shuffle(seed=42).select(range(EVAL_SIZE))

# Split train into train/val
split       = train_small.train_test_split(test_size=0.15, seed=42)
train_ds    = split['train']
val_ds      = split['test']

print(f'Fine-tune train : {len(train_ds)}')
print(f'Validation      : {len(val_ds)}')
print(f'Test            : {len(test_small)}')

# Label distribution
labels = [ex['label'] for ex in train_ds]
dist   = pd.Series(labels).map(dict(enumerate(LABEL_NAMES))).value_counts()
print(f'\nLabel distribution:\n{dist}')

## Step 4: EDA — Visualise the Dataset

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class distribution
dist.plot(kind='bar', ax=axes[0], color=['#3498db','#2ecc71','#e74c3c','#9b59b6'],
          edgecolor='white', rot=0)
axes[0].set_title('AG News — Class Distribution (train subset)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 str(int(bar.get_height())), ha='center', fontsize=9)

# Text length distribution
lengths = [len(ex['text'].split()) for ex in train_ds]
axes[1].hist(lengths, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(np.median(lengths), color='red', linestyle='--',
                label=f'Median: {int(np.median(lengths))} words')
axes[1].set_title('Headline Word-Length Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Words per headline')
axes[1].legend()

plt.tight_layout()
plt.savefig('task1_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Tokenize the Dataset

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False   # DataCollatorWithPadding handles dynamic padding
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=['text'])
test_tok  = test_small.map(tokenize, batched=True, remove_columns=['text'])

# Rename 'label' to 'labels' (what Trainer expects)
train_tok = train_tok.rename_column('label', 'labels')
val_tok   = val_tok.rename_column('label', 'labels')
test_tok  = test_tok.rename_column('label', 'labels')

train_tok.set_format('torch')
val_tok.set_format('torch')
test_tok.set_format('torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenization complete!')
print(f'Columns: {train_tok.column_names}')

## Step 6: Load BERT and Define Metrics

In [ ]:
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

print('Model loaded!')

## Step 7: Fine-Tune BERT

In [ ]:
training_args = TrainingArguments(
    output_dir              = './bert_news_classifier',
    num_train_epochs        = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_ratio            = 0.1,
    weight_decay            = 0.01,
    learning_rate           = 2e-5,
    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    fp16                    = (DEVICE == 'cuda'),
    logging_steps           = 50,
    report_to               = 'none',
    dataloader_num_workers  = 0,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

print('Starting fine-tuning...')
print('(~10-20 min on CPU | ~2-5 min on GPU)')
train_result = trainer.train()
print(f'\nTraining complete!')
print(f'Loss : {train_result.training_loss:.4f}')
print(f'Time : {train_result.metrics["train_runtime"]:.0f}s')

In [ ]:
# Save fine-tuned model
MODEL_DIR = './bert_news_final'
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f'Model saved to {MODEL_DIR}')

## Step 8: Evaluate on Test Set

In [ ]:
preds_out  = trainer.predict(test_tok)
y_pred     = np.argmax(preds_out.predictions, axis=-1)
y_true     = preds_out.label_ids

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='weighted')

print(f'Test Accuracy : {acc:.4f}')
print(f'Test F1 (weighted) : {f1:.4f}')
print(f'\n=== Per-Class Classification Report ===')
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title(f'BERT News Classifier — Confusion Matrix\nAcc={acc:.3f}  F1={f1:.3f}',
          fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('task1_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot training loss curve from trainer logs
logs = [x for x in trainer.state.log_history if 'loss' in x and 'eval_loss' not in x]
if logs:
    steps  = [x['step'] for x in logs]
    losses = [x['loss'] for x in logs]
    plt.figure(figsize=(9, 4))
    plt.plot(steps, losses, color='steelblue', linewidth=1.5)
    plt.title('Training Loss Curve', fontsize=12, fontweight='bold')
    plt.xlabel('Step'); plt.ylabel('Loss')
    plt.tight_layout()
    plt.savefig('task1_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 9: Gradio Deployment — Live Interactive Demo

In [ ]:
import gradio as gr

# Load fine-tuned model into a pipeline
classifier = pipeline(
    'text-classification',
    model     = MODEL_DIR,
    tokenizer = MODEL_DIR,
    device    = 0 if DEVICE == 'cuda' else -1,
    top_k     = None    # return scores for all labels
)

EXAMPLE_HEADLINES = [
    'NASA launches new Mars mission to search for ancient life',
    'Stock markets rally as Fed signals rate cut pause',
    'Manchester United wins Premier League title in dramatic final',
    'UN Security Council meets to address Middle East ceasefire',
    'Apple unveils revolutionary AI chip for iPhone 17',
]

def predict_topic(headline: str):
    if not headline.strip():
        return {label: 0.0 for label in LABEL_NAMES}
    results = classifier(headline)[0]  # list of {label, score}
    return {r['label']: round(r['score'], 4) for r in results}

demo = gr.Interface(
    fn          = predict_topic,
    inputs      = gr.Textbox(
                    label       = 'News Headline',
                    placeholder = 'Paste any news headline here...',
                    lines       = 2
                  ),
    outputs     = gr.Label(num_top_classes=4, label='Topic Probabilities'),
    title       = '📰 BERT News Topic Classifier',
    description = 'Fine-tuned BERT (bert-base-uncased) on AG News. '
                  'Classifies headlines into: World · Sports · Business · Sci/Tech',
    examples    = [[h] for h in EXAMPLE_HEADLINES],
    theme       = gr.themes.Soft()
)

print('Launching Gradio app...')
demo.launch(share=False)   # set share=True for a public link

## Step 10: Summary & Key Findings

| Metric | Value |
|--------|-------|
| Base Model | bert-base-uncased (110M params) |
| Dataset | AG News (4,000 train / 800 test subset) |
| Epochs | 3 |
| Test Accuracy | computed above |
| Test F1 (weighted) | computed above |

**Key Insights:**
- BERT's contextual embeddings capture topic-relevant features far better than traditional TF-IDF
- Sports and Sci/Tech headlines are most distinguishable; World vs Business occasionally overlap
- 3 epochs is sufficient for convergence on 4K samples; more data further improves performance
- Gradio provides instant no-code deployment for stakeholder demos

**To improve further:**
- Train on the full 120K AG News training set
- Try `distilbert-base-uncased` for 2× speed with minimal accuracy loss
- Add class-weighted loss if distribution becomes imbalanced